
## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1) A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2) A function or coroutine to execute.

In [1]:

import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. Parrots are part of the Psittaciformes order, and they include species like macaws, cockatiels, and budgies. They\'re known for their ability to mimic human speech, which is pretty unique among birds.\n\nFirst, I should think about the evolutionary reasons. Why would a bird evolve to mimic human speech? Well, maybe in the wild, they mimic other birds or animals. Mimicry is a survival trait—it could help them communicate with their flock, avoid predators by imitating alarm calls, or even find food by copying other birds\' calls. So in the wild, their vocal mimicry is a natural behavior.\n\nNow, when they\'re in captivity, humans expose them to a lot of speech. Parrots have a syrinx, which is their vocal organ, similar to a human larynx but more complex. This allows them to produce a wide range of sounds. Their vocal learning ability is comparable to some mammals

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'fm6067qqw', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.147314295, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.005979769, 'prompt_tokens_details': None, 'queue_time': 0.052806468, 'total_time': 0.153294064}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019eaf6f-48fc-7160-a


## Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. Let me format the tool call correctly.\n', 'tool_calls': [{'id': 'ebkf3rreq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 153, 'total_tokens': 245, 'completion_time': 0.15070998, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.006078925, 'prompt_tokens_details': None, 'queue_time': 0.052869004, 'total_time': 0.156788905}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_rea